# 🕵️ Dia 43 — Escrevendo inteligência de ameaças em STIX 2.1

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 43* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-43.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A analista da CoopVale recebeu o PNG do seu grafo do Dia 40 e perguntou: *"como eu ponho isso no meu SIEM?"*. Rafa resume: um PNG não carrega confiança, nem TLP, nem origem. E "Pescador" é um apelido num comentário de código, não um `threat-actor`.

Neste notebook você vai:

1. **Ler o bundle mínimo do original** e responder às perguntas do passo 1 com código.
2. **Escrever um minivalidador** das regras "MUST" e quebrar o bundle de propósito.
3. **Montar o `cluster-iscakit.json` sem a biblioteca stix2**: dicionários Python + `json` + `uuid`.
4. **Conferir o checkpoint por código** e desenhar o grafo.
5. **Jogar o quiz "indicator ou observable?"**.

🐍 Só biblioteca padrão. A biblioteca `stix2` e o validador oficial aparecem numa célula **opcional**.

## 1. O bundle mínimo, lido por código

É o mesmo JSON do passo 1 do original, que passou no `stix2_validator` sem erros. Vamos carregá-lo e responder às perguntas do original.

In [ ]:
import json            # STIX é JSON: a biblioteca padrão já lê e escreve
import uuid            # IDs STIX = tipo + "--" + UUID
from datetime import datetime, timezone

MINIMO = json.loads(r"""{
  "type": "bundle",
  "id": "bundle--ee1feaeb-4d87-4743-99f8-8b8dff5ec362",
  "objects": [
    {"type": "identity", "spec_version": "2.1",
     "id": "identity--054213a1-ee0a-4ddc-8ebb-4257cfe6e084",
     "created": "2026-09-30T12:00:00.000Z", "modified": "2026-09-30T12:00:00.000Z",
     "name": "Trainee Analyst (lab)", "identity_class": "individual"},
    {"type": "indicator", "spec_version": "2.1",
     "id": "indicator--54eac790-404d-4a24-9695-b4a96fd71033",
     "created_by_ref": "identity--054213a1-ee0a-4ddc-8ebb-4257cfe6e084",
     "created": "2026-09-30T12:00:00.000Z", "modified": "2026-09-30T12:00:00.000Z",
     "name": "Phishing landing domain from lab seed report",
     "description": "Domain listed in the seed report as hosting a credential phishing page.",
     "indicator_types": ["malicious-activity"],
     "pattern": "[domain-name:value = 'docs-share.example']",
     "pattern_type": "stix",
     "valid_from": "2026-09-30T12:00:00Z",
     "valid_until": "2026-12-31T00:00:00Z",
     "confidence": 60,
     "object_marking_refs": ["marking-definition--34098fce-860f-48ae-8e50-ebd3cc5e41da"]},
    {"type": "malware", "spec_version": "2.1",
     "id": "malware--c2a1012f-5dcf-43a9-aedb-882438e77826",
     "created": "2026-09-30T12:00:00.000Z", "modified": "2026-09-30T12:00:00.000Z",
     "name": "ExampleLoader", "description": "Illustrative downloader family used for this lab.",
     "is_family": true, "malware_types": ["downloader"]},
    {"type": "attack-pattern", "spec_version": "2.1",
     "id": "attack-pattern--f71f51a9-f77e-4eb4-8304-0f0a58410942",
     "created": "2026-09-30T12:00:00.000Z", "modified": "2026-09-30T12:00:00.000Z",
     "name": "Spearphishing Link",
     "external_references": [{"source_name": "mitre-attack", "external_id": "T1566.002", "url": "https://attack.mitre.org/techniques/T1566/002/"}]},
    {"type": "relationship", "spec_version": "2.1",
     "id": "relationship--bc3bf2cd-8f90-4b71-85f6-bf40c7d1d03a",
     "created": "2026-09-30T12:00:00.000Z", "modified": "2026-09-30T12:00:00.000Z",
     "relationship_type": "indicates",
     "source_ref": "indicator--54eac790-404d-4a24-9695-b4a96fd71033",
     "target_ref": "malware--c2a1012f-5dcf-43a9-aedb-882438e77826"},
    {"type": "relationship", "spec_version": "2.1",
     "id": "relationship--dbd0d499-303a-44fa-82a2-4f55de00ce98",
     "created": "2026-09-30T12:00:00.000Z", "modified": "2026-09-30T12:00:00.000Z",
     "relationship_type": "uses",
     "source_ref": "malware--c2a1012f-5dcf-43a9-aedb-882438e77826",
     "target_ref": "attack-pattern--f71f51a9-f77e-4eb4-8304-0f0a58410942"}
  ]
}""")

# Quantos objetos de cada tipo?
from collections import Counter
print("Tipos no bundle:", dict(Counter(o["type"] for o in MINIMO["objects"])))

# Pergunta 1 do original: quais campos o indicador tem?
indicador = next(o for o in MINIMO["objects"] if o["type"] == "indicator")
print("\nCampos do indicador:", sorted(indicador))

# Pergunta 3: o que é marking-definition--34098fce-...?
TLP = {  # marcações TLP 1.0 PREDEFINIDAS no STIX 2.1 (IDs fixos da especificação)
    "marking-definition--613f2e26-407d-48c7-9eca-b8e91df99dc9": "TLP:WHITE",
    "marking-definition--34098fce-860f-48ae-8e50-ebd3cc5e41da": "TLP:GREEN",
    "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82": "TLP:AMBER",
    "marking-definition--5e57c739-391a-4eb3-b6be-7d15ca92d5ed": "TLP:RED",
}
for ref in indicador["object_marking_refs"]:
    print("Marcação do indicador:", TLP.get(ref, "desconhecida (TLP 2.0 precisa de extensão: Dia 44)"))

# Pergunta 4: o formato do ID
tipo, _, u = indicador["id"].partition("--")
print(f"\nID = '{tipo}' + '--' + UUID versão {uuid.UUID(u).version}")

## 2. Um minivalidador das regras "MUST"

O validador oficial checa centenas de regras. Aqui escrevemos só as que o passo 2 do original pede para quebrar, mais algumas do checkpoint. Isso **não substitui** o `stix2_validator`, mas ensina o que ele está olhando.

- campos comuns obrigatórios em todo objeto (exceto o `bundle`): `type`, `spec_version`, `id`, `created`, `modified`;
- no `indicator`: `pattern`, `pattern_type`, `valid_from`;
- `spec_version` precisa ser `"2.1"`;
- `is_family` precisa ser booleano, e com `True` exige `name`;
- `confidence`, se existir, é inteiro de 0 a 100;
- ID = tipo do próprio objeto + `--` + UUID válido;
- toda referência `*_ref` aponta para um objeto do bundle ou para uma marcação TLP predefinida.

In [ ]:
COMUNS = ("type", "spec_version", "id", "created", "modified")
OBRIGATORIOS = {
    "indicator": ("pattern", "pattern_type", "valid_from"),
    "relationship": ("relationship_type", "source_ref", "target_ref"),
    "infrastructure": ("name",),
    "intrusion-set": ("name",),
    "identity": ("name",),
    "attack-pattern": ("name",),
}

def validar(bundle: dict) -> list:
    """Devolve uma lista de erros (vazia = passou nas regras que conhecemos)."""
    erros = []
    objs = bundle.get("objects", [])
    ids = {o.get("id") for o in objs} | set(TLP)        # TLP predefinida pode ser só referenciada
    for o in objs:
        oid = o.get("id", "<sem id>")
        if o.get("type") == "marking-definition":      # marcações seguem regras próprias
            continue
        for campo in COMUNS + OBRIGATORIOS.get(o.get("type"), ()):
            if campo not in o:
                erros.append(f"{oid}: falta '{campo}'")
        if o.get("spec_version") not in (None, "2.1"):
            erros.append(f"{oid}: spec_version '{o['spec_version']}' (precisa ser '2.1')")
        tipo, sep, u = str(o.get("id", "")).partition("--")
        try:
            uuid.UUID(u)
            if tipo != o.get("type") or not sep:
                raise ValueError
        except ValueError:
            erros.append(f"{oid}: ID fora do formato tipo--UUID")
        if "is_family" in o:
            if not isinstance(o["is_family"], bool):
                erros.append(f"{oid}: is_family precisa ser true/false, veio {o['is_family']!r}")
            elif o["is_family"] and "name" not in o:
                erros.append(f"{oid}: is_family=true exige 'name'")
        if "confidence" in o and not (isinstance(o["confidence"], int) and 0 <= o["confidence"] <= 100):
            erros.append(f"{oid}: confidence precisa ser inteiro 0-100")
        for chave, valor in o.items():                  # referências quebradas
            refs = valor if chave.endswith("_refs") else [valor] if chave.endswith("_ref") else []
            for r in refs:
                if r not in ids:
                    erros.append(f"{oid}: {chave} aponta para objeto ausente {r}")
    return erros

print("Bundle mínimo:", validar(MINIMO) or "✅ sem erros nas regras conhecidas")

### Quebre de propósito

O original manda apagar `valid_from`, trocar `is_family` por `"yes"` e mudar `spec_version` para `"2.0"`, rodando o validador depois de cada mudança. Fazemos o mesmo numa cópia.

In [ ]:
import copy

def quebrar(descricao, mexer):
    b = copy.deepcopy(MINIMO)          # nunca mexa no original: trabalhe numa cópia
    mexer(b["objects"])
    print(f"\n🔨 {descricao}")
    for e in validar(b) or ["(nenhum erro detectado: o minivalidador não cobre esta regra)"]:
        print("   ", e)

def por_tipo(objs, t):
    return next(o for o in objs if o["type"] == t)

quebrar("apagar valid_from do indicador", lambda objs: por_tipo(objs, "indicator").pop("valid_from"))
quebrar('is_family = "yes"', lambda objs: por_tipo(objs, "malware").update(is_family="yes"))
quebrar('spec_version = "2.0" no malware', lambda objs: por_tipo(objs, "malware").update(spec_version="2.0"))
quebrar("confidence = 150", lambda objs: por_tipo(objs, "indicator").update(confidence=150))

## 3. ➕ O cluster do IscaKit em STIX, só com stdlib

Os dados vêm de `caso-continuo/dados/dia-43/cluster-pescador.json` (o "Dia 40" da Operação Isca: 10 nós, 11 arestas, tudo fictício). A célula tenta baixar do repositório e, sem rede, usa a cópia embutida.

In [ ]:
import urllib.request

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-43/cluster-pescador.json")
EMBUTIDO = r"""{
 "_aviso": "Dados 100% fictícios da Operação Isca (Investigador Cyber 90 Dias, Dia 43). Domínios .example (RFC 2606), IPs de documentação (RFC 5737). Equivale ao nodes.csv/edges.csv do Dia 40 do original.",
 "semente": {
  "relatorio": "Relatório interno Vigia VIGIA-2026-001, triagem do SMS 'Pix bloqueado'",
  "data": "2026-10-05"
 },
 "mapa_confianca": {
  "forte": 70,
  "media": 40,
  "fraca": 20
 },
 "nos": [
  {
   "id": "N1",
   "tipo": "domain-name",
   "valor": "serraazul-pix.example",
   "status": "semente",
   "origem": "SMS recebidos pelos cooperados em 2026-10-05 (07h12-08h40 BRT)",
   "forca": "forte"
  },
  {
   "id": "N2",
   "tipo": "ipv4-addr",
   "valor": "203.0.113.77",
   "status": "semente",
   "origem": "Resolução DNS de N1 em 2026-10-05",
   "forca": "forte"
  },
  {
   "id": "N3",
   "tipo": "domain-name",
   "valor": "serraazul-seguranca.example",
   "status": "membro",
   "origem": "Mesmo certificado TLS de N1 (SAN lista os dois nomes), crt.sh",
   "forca": "forte"
  },
  {
   "id": "N4",
   "tipo": "domain-name",
   "valor": "coopserra-azul.example",
   "status": "nao-confirmado",
   "origem": "Só semelhança de nome; registrado em 2026-10-03, página estacionada, sem kit",
   "forca": "fraca"
  },
  {
   "id": "N5",
   "tipo": "domain-name",
   "valor": "coopvale-pix.example",
   "status": "membro",
   "origem": "Mesmo hash de corpo HTML do IscaKit (3 resultados) e mesmo IP 203.0.113.77 em DNS passivo",
   "forca": "media"
  },
  {
   "id": "N6",
   "tipo": "domain-name",
   "valor": "credminas-seguranca.example",
   "status": "membro",
   "origem": "Mesmo favicon customizado (hash -1029384756, 9 resultados) e mesma string '// pescador' no JS em varredura pública",
   "forca": "media"
  },
  {
   "id": "N7",
   "tipo": "ipv4-addr",
   "valor": "198.51.100.40",
   "status": "membro",
   "origem": "Resolução DNS de N6 e N8",
   "forca": "media"
  },
  {
   "id": "N8",
   "tipo": "domain-name",
   "valor": "unicoop-sul-pix.example",
   "status": "membro",
   "origem": "Certificado TLS com SAN compartilhado com N6, crt.sh",
   "forca": "forte"
  },
  {
   "id": "N9",
   "tipo": "domain-name",
   "valor": "cooperouro-regulariza.example",
   "status": "nao-confirmado",
   "origem": "Só o mesmo título de página 'Regularização de Chave Pix' (título genérico, 214 resultados)",
   "forca": "fraca"
  },
  {
   "id": "N10",
   "tipo": "url",
   "valor": "https://lnk.example/sa7",
   "status": "semente",
   "origem": "Encurtador citado no SMS; redireciona para N1",
   "forca": "forte"
  }
 ],
 "arestas": [
  {
   "de": "N1",
   "para": "N2",
   "pivo": "DNS A (resolução)",
   "resultados": 1,
   "forca": "forte"
  },
  {
   "de": "N1",
   "para": "N3",
   "pivo": "SAN do mesmo certificado TLS (crt.sh)",
   "resultados": 2,
   "forca": "forte"
  },
  {
   "de": "N1",
   "para": "N4",
   "pivo": "semelhança lexical do nome (sem pivô técnico)",
   "resultados": 0,
   "forca": "fraca"
  },
  {
   "de": "N1",
   "para": "N5",
   "pivo": "hash do corpo HTML do IscaKit",
   "resultados": 3,
   "forca": "media"
  },
  {
   "de": "N5",
   "para": "N2",
   "pivo": "DNS passivo: mesmo IP",
   "resultados": 41,
   "forca": "fraca"
  },
  {
   "de": "N1",
   "para": "N6",
   "pivo": "hash de favicon customizado",
   "resultados": 9,
   "forca": "media"
  },
  {
   "de": "N6",
   "para": "N7",
   "pivo": "DNS A (resolução)",
   "resultados": 1,
   "forca": "forte"
  },
  {
   "de": "N6",
   "para": "N8",
   "pivo": "SAN do mesmo certificado TLS (crt.sh)",
   "resultados": 2,
   "forca": "forte"
  },
  {
   "de": "N8",
   "para": "N7",
   "pivo": "DNS A (resolução)",
   "resultados": 1,
   "forca": "forte"
  },
  {
   "de": "N1",
   "para": "N9",
   "pivo": "mesmo título de página",
   "resultados": 214,
   "forca": "fraca"
  },
  {
   "de": "N10",
   "para": "N1",
   "pivo": "redirecionamento HTTP observado",
   "resultados": 1,
   "forca": "forte"
  }
 ]
}"""

try:
    with urllib.request.urlopen(URL, timeout=10) as r:
        CASO = json.loads(r.read().decode("utf-8"))
    print("Cluster baixado do repositório.")
except Exception as erro:
    CASO = json.loads(EMBUTIDO)          # plano B offline: mesma informação
    print(f"Sem rede ({type(erro).__name__}): usando a cópia embutida.")

for n in CASO["nos"]:
    print(f"  {n['id']:<4} {n['tipo']:<12} {n['valor']:<34} {n['status']:<15} força={n['forca']}")

### As decisões do analista, como parâmetros

O mapeamento de confiança e o destino dos nós não confirmados são **decisões suas**, então ficam num formulário. Os valores padrão seguem a sugestão do README.

In [ ]:
#@title Decisões de codificação { display-mode: "form" }
CONF_FORTE = 70  #@param {type:"integer"}
CONF_MEDIA = 40  #@param {type:"integer"}
CONF_FRACA = 20  #@param {type:"integer"}
INCLUIR_N4_COM_CONF = 15  #@param {type:"integer"}
INCLUIR_N9 = False  #@param {type:"boolean"}

MAPA = {"forte": CONF_FORTE, "media": CONF_MEDIA, "fraca": CONF_FRACA}
print("Mapeamento escrito (vai na nota de 3 linhas):", MAPA)
print("N4 (coopserra-azul.example) entra com confiança", INCLUIR_N4_COM_CONF, "| N9 entra?", INCLUIR_N9)

In [ ]:
# Momento único para todos os objetos: o mesmo "created" deixa o bundle coerente.
AGORA = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
VALIDO_DE = "2026-10-05T00:00:00Z"        # dia do disparo dos SMS
VALIDO_ATE = "2026-12-31T00:00:00Z"       # indicadores de phishing envelhecem rápido
TLP_AMBER = "marking-definition--f88d31f6-486f-44da-b317-01333bde0b82"   # predefinida (TLP 1.0)

def novo_id(tipo):
    """Objetos que NÓS criamos usam UUID versão 4 (aleatório), como pede o original."""
    return f"{tipo}--{uuid.uuid4()}"

def base(tipo, **campos):
    """Campos comuns + os específicos. Todo SDO nasce com autor e marcação."""
    obj = {"type": tipo, "spec_version": "2.1", "id": novo_id(tipo),
           "created": AGORA, "modified": AGORA}
    obj.update(campos)
    return obj

vigia = base("identity", name="Vigia (MSSP fictício) - time de investigação", identity_class="organization")
autor = {"created_by_ref": vigia["id"], "object_marking_refs": [TLP_AMBER]}

infra = base("infrastructure", name="Cluster IscaKit (Operação Isca)",
             infrastructure_types=["phishing"],
             description=("Páginas de captura de credenciais imitando cooperativas de crédito, "
                          "servidas pelo mesmo kit (IscaKit). Artefato textual do kit: string "
                          "'// pescador' no JavaScript. Nenhuma afirmação sobre quem opera."),
             **autor)

def padrao(no):
    """Converte o nó do grafo para a linguagem de padrões STIX."""
    campo = {"domain-name": "domain-name:value", "ipv4-addr": "ipv4-addr:value", "url": "url:value"}[no["tipo"]]
    return f"[{campo} = '{no['valor']}']"

# Índice das arestas por nó, para pôr o pivô e a contagem na descrição do relacionamento.
pivos = {}
for a in CASO["arestas"]:
    for ponta in (a["de"], a["para"]):
        pivos.setdefault(ponta, []).append(a)

# O CLUSTER de atividade: intrusion-set (agrupamento), NÃO threat-actor (pessoas).
cluster = base("intrusion-set", name="CA-ISCA-01 (cluster de atividade do IscaKit)",
               description=("Agrupamento de atividade definido pela Vigia a partir de infraestrutura e kit "
                            "compartilhados. Não identifica pessoa nem organização."),
               confidence=MAPA["media"], **autor)

objetos, rels, deixados_de_fora = [vigia, cluster, infra], [], []
rels.append(base("relationship", relationship_type="uses", source_ref=cluster["id"], target_ref=infra["id"],
                 description="O cluster é definido por esta infraestrutura.", **autor))
for no in CASO["nos"]:
    if no["status"] == "nao-confirmado":
        if no["id"] == "N4" and INCLUIR_N4_COM_CONF:
            conf = INCLUIR_N4_COM_CONF
        elif no["id"] == "N9" and INCLUIR_N9:
            conf = MAPA["fraca"]
        else:
            deixados_de_fora.append(no)
            continue
    else:
        conf = MAPA[no["forca"]]
    ind = base("indicator",
               name=f"{no['tipo']} do cluster IscaKit: {no['valor'].replace('.', '[.]')}",
               description=f"Origem: {no['origem']}. Status no grafo: {no['status']}.",
               indicator_types=["malicious-activity"],
               pattern=padrao(no), pattern_type="stix",
               valid_from=VALIDO_DE, valid_until=VALIDO_ATE,
               confidence=conf, **autor)
    objetos.append(ind)
    desc = "; ".join(f"pivô: {a['pivo']} ({a['resultados']} resultados, {a['forca']})" for a in pivos.get(no["id"], []))
    rels.append(base("relationship", relationship_type="indicates",
                     source_ref=ind["id"], target_ref=infra["id"],
                     description=desc or "nó semente", confidence=conf, **autor))

# Técnicas ATT&CK (IDs de objeto aleatórios aqui; no ATT&CK real você usaria o ID do objeto da MITRE)
for tid, nome, url in [("T1566.002", "Spearphishing Link", "https://attack.mitre.org/techniques/T1566/002/"),
                       ("T1660", "Phishing (Mobile)", "https://attack.mitre.org/techniques/T1660/")]:
    ap = base("attack-pattern", name=nome,
              external_references=[{"source_name": "mitre-attack", "external_id": tid, "url": url}])
    objetos.append(ap)
    rels.append(base("relationship", relationship_type="uses", source_ref=cluster["id"],
                     target_ref=ap["id"], description="SMS com link para a página do kit (Operação Isca).", **autor))

BUNDLE = {"type": "bundle", "id": novo_id("bundle"), "objects": objetos + rels}

with open("cluster-iscakit.json", "w", encoding="utf-8") as f:
    json.dump(BUNDLE, f, ensure_ascii=False, indent=2)

print(len(BUNDLE["objects"]), "objetos gravados em cluster-iscakit.json")
print("Deixados de fora:", [f"{n['id']} {n['valor']}" for n in deixados_de_fora])
print("Validação:", validar(BUNDLE) or "✅ sem erros nas regras conhecidas")

## 4. O checkpoint, conferido por código

In [ ]:
objs = BUNDLE["objects"]
inds = [o for o in objs if o["type"] == "indicator"]

checagens = {
    "Nenhum objeto é threat-actor": all(o["type"] != "threat-actor" for o in objs),
    "Nenhum relacionamento é attributed-to": all(o.get("relationship_type") != "attributed-to" for o in objs),
    "Toda description de indicador diz a origem": all(i.get("description", "").startswith("Origem:") for i in inds),
    "Todo indicador tem confiança do mapeamento (ou a exceção escrita)":
        all(i["confidence"] in set(MAPA.values()) | {INCLUIR_N4_COM_CONF} for i in inds),
    "Todo SDO criado por nós tem marcação TLP":
        all(o.get("object_marking_refs") for o in objs if o["type"] not in ("identity", "attack-pattern")),
    "'Pescador' não é nome de nenhum objeto": all("pescador" not in o.get("name", "").lower() for o in objs),
}
for frase, ok in checagens.items():
    print("✅" if ok else "❌", frase)

In [ ]:
# O grafo do bundle em texto (o que o STIX visualizer desenharia).
nomes = {o["id"]: o.get("name", o["type"]) for o in objs}
print("Grafo do bundle (relationship objects):\n")
for r in (o for o in objs if o["type"] == "relationship"):
    print(f"  [{nomes[r['source_ref']][:52]:<52}] --{r['relationship_type']}--> [{nomes[r['target_ref']][:34]}]")

# E um mermaid para colar no seu README do GitHub (ele renderiza).
linhas = ["graph LR"]
apelido = {oid: f"n{i}" for i, oid in enumerate(nomes)}
for r in (o for o in objs if o["type"] == "relationship"):
    s, t = r["source_ref"], r["target_ref"]
    linhas.append(f'    {apelido[s]}["{nomes[s][:40]}"] -- "{r["relationship_type"]}" --> {apelido[t]}["{nomes[t][:40]}"]')
with open("cluster-iscakit.mmd", "w", encoding="utf-8") as f:
    f.write("\n".join(linhas) + "\n")
print("\n💾 Diagrama mermaid salvo em cluster-iscakit.mmd")

## 5. 🎮 Quiz: indicator ou observable?

Para cada situação, responda **1** se o certo é um `indicator` (julgamento de que vale detectar), **2** se é um observável/SCO (fato) ou **3** se não deveria entrar no bundle de hoje.

`MODO_JOGO = False` mostra só o gabarito.

In [ ]:
SITUACOES = [
    ("O domínio serraazul-pix.example existe e resolveu para 203.0.113.77 em 5/out.", "2",
     "É um fato observado. O julgamento ('vale detectar') vai no indicador."),
    ("Queremos que os SIEMs das cooperativas alertem em qualquer acesso a serraazul-pix.example.", "1",
     "É o julgamento de que o padrão merece detecção, com confiança e validade."),
    ("O cluster é operado por uma pessoa chamada 'Pescador'.", "3",
     "Isso seria threat-actor + attributed-to: uma afirmação de atribuição que não temos."),
    ("O IP 198.51.100.40 serviu duas páginas do kit e deve ser bloqueado até dezembro.", "1",
     "Julgamento com validade (valid_until) e confiança."),
    ("Um certificado TLS com SAN listando serraazul-pix.example e serraazul-seguranca.example.", "2",
     "x509-certificate é um SCO: o fato de que o certificado existe."),
    ("O nome do cooperado que caiu no golpe e transferiu via Pix.", "3",
     "Dado pessoal de vítima. Não tem lugar num bundle de compartilhamento."),
    ("cooperouro-regulariza.example tem o mesmo título de página que outros 213 sites.", "3",
     "Pivô fraco demais (título genérico). Fica fora, ou entra com confiança ≤ 20 e justificativa."),
]

MODO_JOGO = True

acertos = 0
for i, (sit, certo, porque) in enumerate(SITUACOES, start=1):
    print(f"\n[{i}/{len(SITUACOES)}] {sit}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3"):
            r = input("   1=indicator  2=observable  3=fora do bundle: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {dict(zip('123', ('indicator', 'observable', 'fora do bundle')))[certo]} — {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(SITUACOES)}")

## 6. (Opcional) O validador oficial

Se quiser validar com a ferramenta da OASIS, rode a célula abaixo no Colab. A versão fixada é a mesma do original (3.2.0), por causa do problema de *schema* da 3.3.1.

In [ ]:
!pip -q install "stix2-validator==3.2.0"
!stix2_validator cluster-iscakit.json

# Fora do Colab (ou sem as linhas com "!"), tentamos pela API Python, se a lib existir.
try:
    from stix2validator import validate_file, print_results
    print_results(validate_file("cluster-iscakit.json"))
except ImportError:
    print("stix2-validator não instalado: tudo bem, o minivalidador acima cobriu o essencial.")

In [ ]:
# Leve o bundle para o seu fork.
try:
    from google.colab import files
    files.download("cluster-iscakit.json")
    files.download("cluster-iscakit.mmd")
except ImportError:
    print("(Fora do Colab: cluster-iscakit.json e cluster-iscakit.mmd estão na pasta atual)")

## ✅ Antes de fechar o Dia 43

- [ ] O minivalidador acusou os três erros de propósito do original
- [ ] `cluster-iscakit.json` gerado, sem `threat-actor` e sem `attributed-to`
- [ ] Nota de três linhas com o seu mapeamento forte/média/fraca → confiança
- [ ] Justificativa escrita para N4 e N9
- [ ] (Original) `cluster-a.json` validado com `stix2_validator` e captura do STIX visualizer

**Amanhã (Dia 44):** o bundle vai sair da Vigia. Para quem? Com qual TLP? Você puxa dados de um servidor TAXII real e escreve o registro de decisão de compartilhamento.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>